In [4]:
import torch
from torch_geometric.data import Data

# 1. Load the pre-saved PyTorch tensor files
edge_index = torch.load(r"C:\Users\Administrator\Desktop\Research\Social media mining research\project\data\edge_index.pt")      # [2, num_edges]
edge_type = torch.load(r"C:\Users\Administrator\Desktop\Research\Social media mining research\project\data\edge_type.pt")        # Relation types (if using Relational GCN/GAT)
edge_weight = torch.load(r"C:\Users\Administrator\Desktop\Research\Social media mining research\project\data\edge_weight.pt")    # [num_edges]
x = torch.load(r"C:\Users\Administrator\Desktop\Research\Social media mining research\project\data\features.pt")                 # Node feature matrix [num_nodes, num_features]
y_bot = torch.load(r"C:\Users\Administrator\Desktop\Research\Social media mining research\project\data\\labels_bot.pt")           # Target label for bot detection [num_nodes]

# Optional: Convert multi-relational graph to a homogeneous edge index if needed
# or keep edge_type for RGCN encoders.

data = Data(
    x=x,
    edge_index=edge_index,
    edge_attr=edge_weight,
    edge_type=edge_type,
    y=y_bot
)

print(data)



Data(x=[10199, 788], edge_index=[2, 1700108], edge_attr=[1700108], y=[10199], edge_type=[1700108])


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, RGCNConv

# --- 1. Graph Augmentation Functions ---
def drop_edges(edge_index, p=0.2):
    mask = torch.rand(edge_index.size(1)) > p
    return edge_index[:, mask]

def mask_features(x, p=0.2):
    mask = torch.rand_like(x) < p
    x_aug = x.clone()
    x_aug[mask] = 0
    return x_aug

# --- 2. GNN Encoder ---
class GNNEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GNNEncoder, self).__init__()
        # If using edge_type, swap GCNConv for RGCNConv
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x

# --- 3. Projection Head for Contrastive Learning ---
class ProjectionHead(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ProjectionHead, self).__init__()
        self.fc1 = nn.Linear(in_channels, out_channels)
        self.fc2 = nn.Linear(out_channels, out_channels)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

# --- 4. Main BotGCL Architecture ---
class BotGCL(nn.Module):
    def __init__(self, in_dim, hidden_dim, proj_dim, num_classes):
        super(BotGCL, self).__init__()
        self.encoder = GNNEncoder(in_dim, hidden_dim, proj_dim)
        self.projection_head = ProjectionHead(proj_dim, proj_dim)
        self.classifier = nn.Linear(proj_dim, num_classes)

    def forward(self, x, edge_index):
        z = self.encoder(x, edge_index)
        logits = self.classifier(z)
        return logits, z

    def contrastive_loss(self, z1, z2, temperature=0.5):
        z1 = F.normalize(self.projection_head(z1), dim=1)
        z2 = F.normalize(self.projection_head(z2), dim=1)
        sim_matrix = torch.matmul(z1, z2.T) / temperature
        labels = torch.arange(z1.size(0), device=z1.device)
        return F.cross_entropy(sim_matrix, labels)

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = data.to(device)

model = BotGCL(
    in_dim=data.x.size(1),
    hidden_dim=128,
    proj_dim=64,
    num_classes=2 # Binary classification: Bot vs Human
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()
lambda_cl = 0.1 # Weight hyperparameter for contrastive loss

def train_epoch(data):
    model.train()
    optimizer.zero_grad()

    # Create two augmented views for contrastive learning
    x1, edge1 = mask_features(data.x, p=0.15), drop_edges(data.edge_index, p=0.15)
    x2, edge2 = mask_features(data.x, p=0.15), drop_edges(data.edge_index, p=0.15)

    # Compute contrastive loss between views
    _, z1 = model(x1, edge1)
    _, z2 = model(x2, edge2)
    loss_cl = model.contrastive_loss(z1, z2)

    # Primary supervised task on original graph
    logits, _ = model(data.x, data.edge_index)
    
    # Filter masked/unlabeled nodes if applicable
    labeled_mask = (data.y != -1) # Assuming -1 or specific mask array
    loss_ce = criterion(logits[labeled_mask], data.y[labeled_mask].long())

    total_loss = loss_ce + lambda_cl * loss_cl
    total_loss.backward()
    optimizer.step()

    return total_loss.item()

for epoch in range(1, 101):
    loss = train_epoch(data)
    if epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | Total Loss: {loss:.4f}")

Epoch 010 | Total Loss: 1.4257
Epoch 020 | Total Loss: 1.3359
Epoch 030 | Total Loss: 1.2915
Epoch 040 | Total Loss: 1.2644
Epoch 050 | Total Loss: 1.2435
Epoch 060 | Total Loss: 1.2282
Epoch 070 | Total Loss: 1.2143
Epoch 080 | Total Loss: 1.2066
Epoch 090 | Total Loss: 1.2043
Epoch 100 | Total Loss: 1.1972
